# Composite Index (EFCI)

Combines recurrence, severity, persistence and ENSO magnitude into the composite chronicity index, classifies it with natural breaks (Jenks) and runs the first sensitivity analyses (ENSO magnitude version, alternative weights, robustness to outliers).

Outputs: `efci_composite_*` tables in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Efficient reader for GeoPackage layers
import pyogrio
# Natural-breaks (Jenks) classification
import jenkspy
# Spearman rank correlation
from scipy.stats import spearmanr

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step8_efci_composite", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("índice compuesto iniciada — configuración cargada desde %s", CONFIG_PATH)

# Composite index settings from the config
EFCI_CFG = config["efci_composite"]
# Base weights of the four components
WEIGHTS_BASE = EFCI_CFG["weights"]
# Check the base weights add up to 1
assert abs(sum(WEIGHTS_BASE.values()) - 1.0) < 1e-9, "los pesos base del EFCI deben sumar 1.0"
# Primary ENSO magnitude column (mean)
MAG_COL_PRIMARY = EFCI_CFG["magnitude_column_primary"]
# Alternative ENSO magnitude column (maximum)
MAG_COL_ALT = EFCI_CFG["magnitude_column_alternative"]
# Number of classes
N_CLASSES = EFCI_CFG["n_classes"]
# Class labels, from lowest to highest
CLASS_LABELS = EFCI_CFG["class_labels"]
# Alternative weight scenarios for the sensitivity analysis
WEIGHT_SCENARIOS = EFCI_CFG["sensitivity_weight_scenarios"]
# Percentile at which severity is capped for the robustness test
WINSOR_PCTL = EFCI_CFG["severity_winsorize_percentile"]
# Log weights, magnitude column and classification settings
logger.info("Pesos base: %s | Magnitud primaria: %s | Clasificación: %s (%d clases)",
            WEIGHTS_BASE, MAG_COL_PRIMARY, EFCI_CFG["classification_method"], N_CLASSES)

## 1. Load and join the four indices

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Keys that identify a canton view
KEYS_CANTON = ["province_code", "province_name_official", "canton_code", "canton_name_official", "hazard_type"]
# Keys that identify a province view
KEYS_PROVINCIA = ["province_code", "province_name_official", "hazard_type"]


# Load and join the four indices for one administrative level
def cargar_y_unir(nivel: str, keys: list[str]) -> pd.DataFrame:
    """Load the four component indices for one administrative level and join them.
    
    Args:
        nivel: 'canton' or 'provincia'.
        keys: Columns that identify a unit and hazard view.
    
    Returns:
        Table with recurrence, severity, persistence and ENSO magnitude per unit.
    """
    # Load the severity index
    sev = pd.read_parquet(processed_dir / f"severity_index_{nivel}.parquet")
    # Load the recurrence index
    rec = pd.read_parquet(processed_dir / f"recurrence_index_{nivel}.parquet")
    # Load the persistence index
    per = pd.read_parquet(processed_dir / f"persistence_index_{nivel}.parquet")
    # Load the ENSO magnitude index
    mag = pd.read_parquet(processed_dir / f"enso_magnitude_index_{nivel}.parquet")

    # Cross-check the event counts of two indices
    check = sev[keys + ["n_eventos"]].merge(per[keys + ["n_eventos"]], on=keys, suffixes=("_sev", "_per"))
    # Check both indices count the same events
    assert (check["n_eventos_sev"] == check["n_eventos_per"]).all(), \
        "n_eventos no coincide entre Severidad y Persistencia — revisar las tablas de origen"

    # Join the four indices on the unit keys
    out = sev[keys + ["n_eventos", "severity_index"]].merge(
        rec[keys + ["recurrence_index"]], on=keys, how="inner"
    ).merge(
        per[keys + ["persistence_index"]], on=keys, how="inner"
    ).merge(
        mag[keys + [MAG_COL_PRIMARY, MAG_COL_ALT]], on=keys, how="inner"
    )
    # Return the per-unit table
    return out


# Join the indices at canton level
efci_canton = cargar_y_unir("canton", KEYS_CANTON)
# Join the indices at province level
efci_provincia = cargar_y_unir("provincia", KEYS_PROVINCIA)
# Log the size of both tables
logger.info("Unión completa — cantón: %d filas | provincia: %d filas", len(efci_canton), len(efci_provincia))

## 2. Base composite index

In [ ]:
# Compute the composite index with a set of weights
def calcular_efci(df: pd.DataFrame, weights: dict[str, float], mag_col: str, out_col: str = "efci") -> pd.DataFrame:
    """Compute the composite index as a weighted sum of the four components.
    
    Args:
        df: Table with the four component columns.
        weights: Weight per component; must add up to 1.
        mag_col: ENSO magnitude column to use.
        out_col: Name of the output column.
    
    Returns:
        Copy of the table with the composite index in [0, 1].
    """
    # Work on a copy
    df = df.copy()
    # Weighted sum of recurrence, severity, persistence and ENSO magnitude
    df[out_col] = (
        weights["recurrencia"] * df["recurrence_index"]
        + weights["severidad"] * df["severity_index"]
        + weights["persistencia"] * df["persistence_index"]
        + weights["magnitud_enso"] * df[mag_col]
    )
    # Check the index is between 0 and 1
    assert df[out_col].between(0, 1).all(), f"{out_col} fuera de [0,1]"
    # Return the table with the index
    return df


# Base composite index at canton level
efci_canton = calcular_efci(efci_canton, WEIGHTS_BASE, MAG_COL_PRIMARY, "efci")
# Base composite index at province level
efci_provincia = calcular_efci(efci_provincia, WEIGHTS_BASE, MAG_COL_PRIMARY, "efci")
# Log that the base index was computed
logger.info("EFCI base calculado (mag=%s) para ambos niveles", MAG_COL_PRIMARY)

## 3. Natural-breaks (Jenks) classification

In [ ]:
# Classify values with Jenks natural breaks
def clasificar_jenks(serie: pd.Series, n_classes: int, labels: list[str]) -> pd.Series:
    """Classify values with Jenks natural breaks.
    
    Args:
        serie: Values to classify.
        n_classes: Number of classes.
        labels: Class labels, from lowest to highest.
    
    Returns:
        The class of each value.
    """
    # Compute the class boundaries
    breaks = jenkspy.jenks_breaks(serie.values, n_classes=n_classes)
    # Convert the boundaries to a list
    breaks = list(breaks)
    # Widen the lowest boundary so the minimum value is included
    breaks[0] -= 1e-9
    # Assign each value to its class
    return pd.cut(serie, bins=breaks, labels=labels, include_lowest=True)


# Select the combined hazard view at canton level
mask_todos_canton = efci_canton["hazard_type"] == "TODOS"
# Classify the canton index
efci_canton.loc[mask_todos_canton, "clase_efci"] = clasificar_jenks(
    efci_canton.loc[mask_todos_canton, "efci"], N_CLASSES, CLASS_LABELS
)

# Select the combined hazard view at province level
mask_todos_prov = efci_provincia["hazard_type"] == "TODOS"
# Classify the province index
efci_provincia.loc[mask_todos_prov, "clase_efci"] = clasificar_jenks(
    efci_provincia.loc[mask_todos_prov, "efci"], N_CLASSES, CLASS_LABELS
)

# Print a header for the class distribution
print("Distribución de clases EFCI (cantón, Jenks):")
# Print how many cantons fall in each class
print(efci_canton.loc[mask_todos_canton, "clase_efci"].value_counts().reindex(CLASS_LABELS))

## 4. Sensitivity analysis

In [ ]:
# Compare a base index against an alternative
def resumen_sensibilidad(base: pd.Series, alterna: pd.Series, clase_base: pd.Series, clase_alterna: pd.Series, nombre: str) -> dict:
    """Compare a base index with an alternative version.
    
    Args:
        base: Base index values.
        alterna: Alternative index values.
        clase_base: Base classes.
        clase_alterna: Alternative classes.
        nombre: Name of the test.
    
    Returns:
        Dictionary with the Spearman correlation and the number of units that change class.
    """
    # Spearman correlation between both versions
    rho, p = spearmanr(base, alterna)
    # Number of units that change class
    cambios = (clase_base.values != clase_alterna.values).sum()
    # Dictionary with the comparison results
    resultado = {
        "prueba": nombre, "spearman_rho": round(rho, 4), "p_valor": round(p, 6),
        "cantones_cambian_clase": int(cambios), "pct_cambian_clase": round(100 * cambios / len(base), 1),
    }
    # Print the correlation and the number of class changes
    print(f"{nombre:45s} rho={rho:.4f}  cantones que cambian de clase: {cambios}/{len(base)} ({resultado['pct_cambian_clase']}%)")
    # Return the comparison results
    return resultado


# List that collects all sensitivity tests
resultados_sensibilidad = []

### 4.1 ENSO magnitude: mean (base) versus maximum

In [ ]:
# Composite index using the maximum ENSO magnitude
efci_canton = calcular_efci(efci_canton, WEIGHTS_BASE, MAG_COL_ALT, "efci_mag_max")
# Classify the alternative index
efci_canton.loc[mask_todos_canton, "clase_efci_mag_max"] = clasificar_jenks(
    efci_canton.loc[mask_todos_canton, "efci_mag_max"], N_CLASSES, CLASS_LABELS
)
# Compare it against the base index
resultados_sensibilidad.append(resumen_sensibilidad(
    efci_canton.loc[mask_todos_canton, "efci"], efci_canton.loc[mask_todos_canton, "efci_mag_max"],
    efci_canton.loc[mask_todos_canton, "clase_efci"], efci_canton.loc[mask_todos_canton, "clase_efci_mag_max"],
    "Magnitud: mean vs. max",
))

### 4.2 Alternative weights

In [ ]:
# Loop over the alternative weight scenarios
for nombre_escenario, pesos in WEIGHT_SCENARIOS.items():
    # Check each scenario's weights add up to 1
    assert abs(sum(pesos.values()) - 1.0) < 1e-9, f"pesos del escenario {nombre_escenario} deben sumar 1.0"
    # Name of the scenario column
    col = f"efci_pesos_{nombre_escenario}"
    # Composite index with the scenario weights
    efci_canton = calcular_efci(efci_canton, pesos, MAG_COL_PRIMARY, col)
    # Name of the scenario class column
    clase_col = f"clase_{col}"
    # Classify the scenario index
    efci_canton.loc[mask_todos_canton, clase_col] = clasificar_jenks(
        efci_canton.loc[mask_todos_canton, col], N_CLASSES, CLASS_LABELS
    )
    # Compare it against the base index
    resultados_sensibilidad.append(resumen_sensibilidad(
        efci_canton.loc[mask_todos_canton, "efci"], efci_canton.loc[mask_todos_canton, col],
        efci_canton.loc[mask_todos_canton, "clase_efci"], efci_canton.loc[mask_todos_canton, clase_col],
        f"Pesos: {nombre_escenario}",
    ))

### 4.3 Severity robustness to the two outliers

In [ ]:
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Weights of the five pillars
PILLAR_WEIGHTS = SEV_CFG["pillar_weights"]

# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"]) + W_INJURED * he["injured"]
)
# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]
# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]
# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]
# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)
# Names of the five pillar score columns
PILLAR_SCORE_COLS = ["score_poblacion", "score_vivienda", "score_agricultura", "score_ganaderia", "score_infraestructura"]

# Copy the event table for the robustness test
he_wins = he.copy()
# Loop over the pillar scores
for col in PILLAR_SCORE_COLS:
    # Cap value: the chosen percentile of the pillar score
    techo = he_wins[col].quantile(WINSOR_PCTL)
    # Cap extreme events at that value
    he_wins[col] = he_wins[col].clip(upper=techo)

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False)[
    ["canton_code", "canton_name_official", "province_name_official"]
].drop_duplicates()
# Attach official names to the events
he_wins = he_wins.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]


# Min-max scaling of one series
def _minmax(s: pd.Series) -> pd.Series:
    # Range of the series
    rango = s.max() - s.min()
    # Min-max scaling of one series
    return pd.Series(0.0, index=s.index) if rango == 0 else (s - s.min()) / rango


# Maximum of each pillar per canton using the capped scores
sev_wins = he_wins.groupby(CANTON_COLS)[PILLAR_SCORE_COLS].max().reset_index()
# Loop over the pillar scores
for col in PILLAR_SCORE_COLS:
    # Normalize each pillar
    sev_wins[col.replace("score_", "norm_")] = _minmax(sev_wins[col])
# Severity index computed from the capped scores
sev_wins["severity_index_robusto"] = (
    PILLAR_WEIGHTS["poblacion"] * sev_wins["norm_poblacion"]
    + PILLAR_WEIGHTS["vivienda"] * sev_wins["norm_vivienda"]
    + PILLAR_WEIGHTS["agricultura"] * sev_wins["norm_agricultura"]
    + PILLAR_WEIGHTS["ganaderia"] * sev_wins["norm_ganaderia"]
    + PILLAR_WEIGHTS["infraestructura"] * sev_wins["norm_infraestructura"]
)
# Log how many cantons have the robust severity
logger.info("Severidad robusta (winsorizada al percentil %.0f) calculada para %d cantones", WINSOR_PCTL * 100, len(sev_wins))

In [ ]:
# Attach the robust severity to the composite table
efci_canton = efci_canton.merge(
    sev_wins[CANTON_COLS + ["severity_index_robusto"]], on=CANTON_COLS, how="left"
)
# Composite index using the robust severity
efci_canton["efci_sev_robusta"] = (
    WEIGHTS_BASE["recurrencia"] * efci_canton["recurrence_index"]
    + WEIGHTS_BASE["severidad"] * efci_canton["severity_index_robusto"]
    + WEIGHTS_BASE["persistencia"] * efci_canton["persistence_index"]
    + WEIGHTS_BASE["magnitud_enso"] * efci_canton[MAG_COL_PRIMARY]
)
# Classify the robust version
efci_canton.loc[mask_todos_canton, "clase_efci_sev_robusta"] = clasificar_jenks(
    efci_canton.loc[mask_todos_canton, "efci_sev_robusta"], N_CLASSES, CLASS_LABELS
)
# Compare it against the base index
resultados_sensibilidad.append(resumen_sensibilidad(
    efci_canton.loc[mask_todos_canton, "efci"], efci_canton.loc[mask_todos_canton, "efci_sev_robusta"],
    efci_canton.loc[mask_todos_canton, "clase_efci"], efci_canton.loc[mask_todos_canton, "clase_efci_sev_robusta"],
    "Severidad: original vs. winsorizada (outliers)",
))

# Select the three cantons affected by the extreme events
casos = efci_canton.loc[mask_todos_canton & efci_canton.canton_name_official.isin(
    ["Distrito Metropolitano de Quito", "Chone", "Baños de Agua Santa"]
)]
# Print the original and robust values for those cantons
print(casos[["canton_name_official", "severity_index", "severity_index_robusto", "efci", "efci_sev_robusta"]].to_string(index=False))

### 4.4 Consolidated sensitivity summary

In [ ]:
# Collect all sensitivity results in one table
tabla_sensibilidad = pd.DataFrame(resultados_sensibilidad)
# Print the sensitivity summary
print(tabla_sensibilidad.to_string(index=False))
# Save the sensitivity summary
tabla_sensibilidad.to_csv(processed_dir / "efci_sensitivity_summary.csv", index=False)
# Log the number of tests
logger.info("Resumen de sensibilidad exportado: %d pruebas", len(tabla_sensibilidad))

## 5. Validation

In [ ]:
# Expected number of cantons
n_cantones_esperados = 221
# Number of cantons in the classified view
n_en_clase = mask_todos_canton.sum()
# Check every canton is classified
assert n_en_clase == n_cantones_esperados, "cobertura incompleta en la clasificación EFCI (cantón)"
# Check every province is classified
assert efci_provincia.loc[mask_todos_prov].shape[0] == 24, "cobertura incompleta en la clasificación EFCI (provincia)"
# Check the composite index is between 0 and 1
assert efci_canton["efci"].between(0, 1).all()
# Check every canton has a class
assert efci_canton["clase_efci"].notna().sum() == n_cantones_esperados == (~efci_canton.loc[mask_todos_canton, "clase_efci"].isna()).sum()
# Log that coverage and range are valid
logger.info("Validación de cobertura y rango del EFCI: OK (221 cantones, 24 provincias, todo en [0,1])")

## 6. Export

In [ ]:
# Tables to export
tablas = {
    "efci_composite_canton": efci_canton,
    "efci_composite_provincia": efci_provincia,
}
# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")
# Print the sensitivity summary table name
print(" - efci_sensitivity_summary")

## 7. Results: national ranking of the index

In [ ]:
# Fifteen cantons with the highest composite index
top15_canton = efci_canton.loc[mask_todos_canton].sort_values("efci", ascending=False).head(15)
# Print a header for the canton ranking
print("Top 15 cantones por EFCI:")
# Print the canton ranking
print(top15_canton[["province_name_official", "canton_name_official", "efci", "clase_efci",
                     "severity_index", "recurrence_index", "persistence_index", MAG_COL_PRIMARY]].to_string(index=False))

# Print an empty line
print()
# Ten provinces with the highest composite index
top10_prov = efci_provincia.loc[mask_todos_prov].sort_values("efci", ascending=False).head(10)
# Print a header for the province ranking
print("Top 10 provincias por EFCI:")
# Print the province ranking
print(top10_prov[["province_name_official", "efci", "clase_efci"]].to_string(index=False))